# Your first neural network — in 30 minutes

By the end of this notebook you will have a working program that recognises handwritten digits. Right at the end you will draw a digit with your mouse and it will read it.

No machine learning experience needed. Almost all the code is written already — your job is to assemble a working network out of the ready pieces, train it and see what happens. There are only a few things to fill in, marked with a pencil.

**How to work:** run the cells from top to bottom with the play button on the left of each cell (or `Shift+Enter`).

**Do this right now:** `File → Save a copy in Drive`. Otherwise your edits will not be saved.

In [ ]:
#@title Run this cell — it downloads the data and sets everything up { display-mode: "form" }
import gzip, struct, urllib.request, os, base64
import numpy as np
import matplotlib.pyplot as plt

MIRRORS = ["https://storage.googleapis.com/cvdf-datasets/mnist/",
           "https://ossci-datasets.s3.amazonaws.com/mnist/"]
FILES = {"train_images": "train-images-idx3-ubyte.gz", "train_labels": "train-labels-idx1-ubyte.gz",
         "test_images": "t10k-images-idx3-ubyte.gz",  "test_labels": "t10k-labels-idx1-ubyte.gz"}

def _get(name):
    if not os.path.exists(name):
        for m in MIRRORS:
            try:
                urllib.request.urlretrieve(m + name, name); break
            except Exception: pass
    with gzip.open(name, "rb") as f:
        magic, count = struct.unpack(">II", f.read(8))
        if magic == 0x803:
            rows, cols = struct.unpack(">II", f.read(8))
            return np.frombuffer(f.read(rows*cols*count), np.uint8).reshape(count, rows*cols)
        return np.frombuffer(f.read(count), np.uint8)

def one_hot(labels):
    y = np.zeros((10, labels.size)); y[labels, np.arange(labels.size)] = 1.0; return y

print("Downloading images of handwritten digits ...")
_raw = {k: _get(v) for k, v in FILES.items()}
X_train = _raw["train_images"].T.astype(np.float64) / 255.0
d_train = _raw["train_labels"].astype(np.int64)
Y_train = one_hot(d_train)
X_test  = _raw["test_images"].T.astype(np.float64) / 255.0
y_test  = _raw["test_labels"].astype(np.int64)
print(f"Ready: {X_train.shape[1]} images to learn from and {X_test.shape[1]} to test on")

# ------------------------------------------- the ready-made pieces the network is built from
def sigmoid(z):
    """sigma(z) = 1 / (1 + e^(-z)). Works on a number or an array of any shape."""
    return 1.0 / (1.0 + np.exp(-np.clip(z, -500, 500)))


def sigmoid_prime(z):
    """sigma'(z) = sigma(z) * (1 - sigma(z))"""
    s = sigmoid(z)
    return s * (1.0 - s)


class _Blank:
    """The ___ placeholder that stands where a number has to be filled in."""
    def __repr__(self): return "___"
___ = _Blank()


def init_network(sizes, seed=0):
    """Creates the network: random weights between neighbouring layers, zero biases."""
    for s in sizes:
        if isinstance(s, _Blank):
            raise ValueError("There is still a ___ placeholder in sizes — replace it with a number")
        if not isinstance(s, (int, np.integer)):
            raise ValueError(f"A layer size must be a whole number, not {s!r}")
    rng = np.random.default_rng(seed)
    weights = [rng.standard_normal((n_out, n_in)) / np.sqrt(n_in)
               for n_in, n_out in zip(sizes[:-1], sizes[1:])]
    biases = [np.zeros((n_out, 1)) for n_out in sizes[1:]]
    return {"sizes": list(sizes), "weights": weights, "biases": biases}


def count_parameters(net):
    return sum(w.size for w in net["weights"]) + sum(b.size for b in net["biases"])


def forward(net, a):
    """Forward pass. a is a (784, m) matrix: one column per image."""
    activations = [a]
    zs = []
    for W, b in zip(net["weights"], net["biases"]):
        z = W @ a + b
        zs.append(z)
        a = sigmoid(z)
        activations.append(a)
    return zs, activations


def predict(net, x):
    """Index of the most active output neuron for each image. Shape of the result: (m,)"""
    _, activations = forward(net, x)
    return np.argmax(activations[-1], axis=0)


def accuracy(net, x, digits):
    """Fraction of correct answers — a number between 0 and 1."""
    return float(np.mean(predict(net, x) == digits))


def backprop(net, x, y):
    """Gradients of the cost for a batch of images."""
    m = x.shape[1]
    zs, activations = forward(net, x)
    L = len(net["weights"])
    grad_w = [None] * L
    grad_b = [None] * L

    # error of the output layer
    delta = (activations[-1] - y) * sigmoid_prime(zs[-1])
    grad_w[-1] = delta @ activations[-2].T / m
    grad_b[-1] = delta.sum(axis=1, keepdims=True) / m

    # drag the error backwards: layer L-1, L-2, ...
    for l in range(2, L + 1):
        delta = (net["weights"][-l + 1].T @ delta) * sigmoid_prime(zs[-l])
        grad_w[-l] = delta @ activations[-l - 1].T / m
        grad_b[-l] = delta.sum(axis=1, keepdims=True) / m

    return grad_w, grad_b


def apply_gradients(net, grad_w, grad_b, eta):
    """One descent step: move all weights and biases against the gradient."""
    for i in range(len(net["weights"])):
        net["weights"][i] -= eta * grad_w[i]
        net["biases"][i] -= eta * grad_b[i]


def FILL_ME(*args, **kwargs):
    raise NotImplementedError(
        "There is still a FILL_ME placeholder here — replace it with the name of the right "
        "function (they are listed in the comment above the line)")

# ------------------------------------------------- drawing pictures and preparing a sketch
def resize_bilinear(img, oh, ow):
    h, w = img.shape
    ys, xs = np.linspace(0, h-1, oh), np.linspace(0, w-1, ow)
    y0, x0 = np.floor(ys).astype(int), np.floor(xs).astype(int)
    y1, x1 = np.minimum(y0+1, h-1), np.minimum(x0+1, w-1)
    wy, wx = (ys-y0)[:, None], (xs-x0)[None, :]
    top = img[np.ix_(y0, x0)]*(1-wx) + img[np.ix_(y0, x1)]*wx
    bot = img[np.ix_(y1, x0)]*(1-wx) + img[np.ix_(y1, x1)]*wx
    return top*(1-wy) + bot*wy

def to_mnist(canvas, box=20, size=28):
    """Crops to the ink, fits it into a 20x20 box and centres it by centre of mass."""
    ink = canvas > 0.05
    if not ink.any(): return np.zeros((size, size))
    r, c = np.where(ink.any(1))[0], np.where(ink.any(0))[0]
    crop = canvas[r[0]:r[-1]+1, c[0]:c[-1]+1]
    h, w = crop.shape
    s = box / max(h, w)
    nh, nw = max(1, round(h*s)), max(1, round(w*s))
    digit = resize_bilinear(crop, nh, nw)
    yy, xx = np.mgrid[0:nh, 0:nw]
    cy, cx = (yy*digit).sum()/digit.sum(), (xx*digit).sum()/digit.sum()
    out = np.zeros((size, size))
    top  = int(np.clip(round(size/2 - cy), 0, size-nh))
    left = int(np.clip(round(size/2 - cx), 0, size-nw))
    out[top:top+nh, left:left+nw] = digit
    return np.clip(out, 0, 1)

def show_pixels(column, label=None):
    """Picture on the left, the same patch as numbers on the right — an image really is numbers."""
    img = column.reshape(28, 28)
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 5))
    a1.imshow(img, cmap="gray_r"); a1.set_xticks([]); a1.set_yticks([])
    a1.set_title("what we see" + (f" (this is a {label})" if label is not None else ""))
    piece = img[9:17, 9:17]
    a2.imshow(piece, cmap="gray_r", vmin=0, vmax=1)
    for r in range(8):
        for c in range(8):
            v = piece[r, c]
            a2.text(c, r, f"{v:.1f}", ha="center", va="center", fontsize=8,
                    color="white" if v > 0.5 else "#666")
    a2.set_xticks([]); a2.set_yticks([])
    a2.set_title("what the computer sees (8x8 patch from the middle)")
    plt.tight_layout(); plt.show()

def show_prediction(net, img, true_digit=None):
    _, acts = forward(net, img.reshape(-1, 1))
    out = acts[-1].ravel(); guess = int(np.argmax(out))
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(9, 3.4), gridspec_kw={"width_ratios": [1, 2]})
    a1.imshow(img.reshape(28, 28), cmap="gray_r"); a1.set_xticks([]); a1.set_yticks([])
    a1.set_title("the image")
    colors = ["#d1495b" if d == guess else "#4c72b0" for d in range(10)]
    a2.barh(np.arange(10), out, color=colors); a2.invert_yaxis()
    a2.set_yticks(range(10)); a2.set_xlim(0, 1)
    a2.set_title("how brightly each of the 10 output neurons fires")
    t = f"the network says: {guess} ({out[guess]*100:.0f}% sure)"
    if true_digit is not None:
        t += "  — correct" if guess == true_digit else f"  — wrong, it is a {true_digit}"
    fig.suptitle(t, fontsize=13); plt.tight_layout(); plt.show()

# ------------------------------------------------------------------------ the two checks
def _need(name):
    f = globals().get(name)
    if f is None: raise AssertionError(f"there is no {name} yet — run the cell above")
    return f

def _lcheck1():
    sizes = _need("sizes")
    assert not any(isinstance(s, _Blank) for s in sizes), "sizes still contains ___ placeholders"
    assert sizes[0] == 784, (f"the input is {sizes[0]}, but it should be 784: a 28x28 image is "
                             "28 * 28 = 784 numbers, one per pixel")
    assert sizes[-1] == 10, (f"the output is {sizes[-1]}, but it should be 10: one neuron for "
                             "each digit from 0 to 9")
    net = _need("net")
    return (f"network {' -> '.join(map(str, sizes))} created, "
            f"with {count_parameters(net):,} adjustable numbers in it")

def _lcheck2():
    train = _need("train")
    probe = init_network([784, 16, 16, 10], seed=7)
    before = accuracy(probe, X_test[:, :2000], y_test[:2000])
    train(probe, X_train[:, :6000], Y_train[:, :6000],
          epochs=2, batch_size=10, eta=3.0, verbose=False)
    after = accuracy(probe, X_test[:, :2000], y_test[:2000])
    assert after > 0.55, (f"the accuracy barely moved ({before:.0%} -> {after:.0%}). "
                          "Check the order: backprop works out where to move the weights first, "
                          "then apply_gradients moves them")
    return f"training works: on a trial run the accuracy went from {before:.0%} to {after:.0%}"

_LCHECKS = {1: ("network built", _lcheck1), 2: ("training loop", _lcheck2)}

def check(step):
    name, fn = _LCHECKS[step]
    try:
        msg = fn()
    except NotImplementedError as e:
        print(f"\u274c Check {step} ({name}): {e}"); return
    except (AssertionError, ValueError) as e:
        print(f"\u274c Check {step} ({name}): {e}"); return
    except Exception as e:
        print(f"\u274c Check {step} ({name}): your code raised {type(e).__name__}: {e}"); return
    print(f"\u2705 Check {step} ({name}): {msg}")

print("All set. Scroll down and run the cells in order.")

---
## Part 1. An image is just numbers

A computer does not see "a two". It sees a 28 × 28 square of pixels, and every pixel is a number between 0 (white) and 1 (black).

That makes **28 · 28 = 784 numbers** per image. That is everything our network gets as input.

Run the cell below. On the left is the picture, on the right is a patch from its middle written out as numbers. Change `i` and run it again to look at other digits.

In [ ]:
i = 0     # pick any number from 0 to 59999 and run the cell again

show_pixels(X_train[:, i], d_train[i])

---
## Part 2. Building the network

A network is a few layers of neurons. The numbers from the image go in on the left, pass through the layers and come out as an answer on the right.

We need to decide how many neurons each layer has:

- **First layer (input).** This is where the pixels land. How many are there in a 28 × 28 image?
- **Two middle layers.** Here the network looks for something of its own, something intermediate. Let's take 16 neurons each — just a sensible number, and we will change it later.
- **Last layer (output).** One neuron per possible answer. How many digits are there?

**Fill in the two numbers in place of `___`.**

In [ ]:
sizes = [___, 16, 16, ___]
#         ^              ^
#   how many pixels   how many different
#   in a 28x28 image  digits can come out

net = init_network(sizes)

print("Network layers:", " -> ".join(str(s) for s in net["sizes"]))
print("Numbers the network will tune:", f"{count_parameters(net):,}")

In [ ]:
check(1)

---
## Part 3. Right now the network can do nothing

All 13 thousand of those numbers are still random. Let's see what the network answers.

The ten bars on the right are the ten output neurons, one per digit. The network answers with whichever digit has the brightest neuron.

In [ ]:
print(f"The untrained network gets {accuracy(net, X_test, y_test)*100:.1f}% of digits right")
print("That is about the same as guessing: ten options, so you hit one in ten.\n")

show_prediction(net, X_test[:, 0], int(y_test[0]))

---
## Part 4. How the network learns

The idea behind learning is simple, and it fits into two actions:

1. The network looks at 10 images and gives its answers. We compare them with the correct ones and get the error.
2. Then we need to work out **which way to move each of those 13 thousand numbers** so that next time the error is a little smaller. That is what `backprop` computes.
3. And finally move them — that is `apply_gradients`.

Then take the next 10 images and repeat. One pass over all 60,000 images makes 6000 such little steps. Each one makes the network slightly better.

**Fill in the names of the two functions in place of `FILL_ME`.** The order matters: first work out where to move, then move.

In [ ]:
def train(net, x, y, epochs=10, batch_size=10, eta=3.0, verbose=True):
    """The training loop. You need to fill in two calls in place of FILL_ME.

    Functions available to you:
        backprop(net, images, answers)  ->  grad_w, grad_b
            looks at the error and works out which way to move every weight
        apply_gradients(net, grad_w, grad_b, eta)
            moves the weights that way; eta sets how big the step is
    """
    rng = np.random.default_rng(0)
    n = x.shape[1]

    for epoch in range(1, epochs + 1):
        order = rng.permutation(n)                      # shuffle the images
        for start in range(0, n, batch_size):
            idx = order[start:start + batch_size]
            batch_x = x[:, idx]                         # 10 images
            batch_y = y[:, idx]                         # 10 correct answers

            # STEP 1: work out which way to move the weights
            grad_w, grad_b = FILL_ME(net, batch_x, batch_y)

            # STEP 2: move the weights that way
            FILL_ME(net, grad_w, grad_b, eta)

        if verbose:
            print(f"Epoch {epoch:2d}: the network recognises {accuracy(net, X_test, y_test)*100:5.2f}% of digits")
    return net

In [ ]:
check(2)

---
## Part 5. Training

Everything is ready now. Run the cell and watch the accuracy climb — it takes less than a minute.

Keep an eye on the first epoch: in a single pass the network jumps from 10% to almost 90%.

In [ ]:
net = init_network(sizes)          # start from a fresh network

train(net, X_train, Y_train, epochs=10, batch_size=10, eta=3.0)

print(f"\nDone. Your network recognises {accuracy(net, X_test, y_test)*100:.2f}% of digits,")
print("and these are images it has never seen before.")

---
## Part 6. Looking at the result

Run it a few times — each run picks a random image.

In [ ]:
i = np.random.randint(X_test.shape[1])
show_prediction(net, X_test[:, i], int(y_test[i]))

## Where it gets things wrong

The mistakes are the interesting part. Look at the bars: in these cases the network is usually torn between two digits, and often they are exactly the two a human would confuse as well.

In [ ]:
errors = np.where(predict(net, X_test) != y_test)[0]
print(f"The network is wrong on {len(errors)} images out of {len(y_test)}\n")

for i in np.random.choice(errors, 2, replace=False):
    show_prediction(net, X_test[:, i], int(y_test[i]))

---
## Part 7. Draw your own digit

Draw with the mouse in the white square, then press **Recognise**.

Before showing your drawing to the network the program adjusts it: crops it to the ink, shrinks it and puts it in the centre. Every image the network learned from looks exactly like that, and without the adjustment it gets confused — try drawing a digit in the corner, for example.

In [ ]:
CANVAS_HTML = """
<canvas id="cnv" width="280" height="280"
        style="border:2px solid #444;border-radius:6px;background:#fff;touch-action:none;cursor:crosshair"></canvas>
<div style="margin-top:8px">
  <button id="btn_clear" style="padding:6px 14px">Clear</button>
  <button id="btn_done"  style="padding:6px 14px;font-weight:bold">Recognise</button>
</div>
<script>
var cnv = document.getElementById('cnv'), ctx = cnv.getContext('2d');
ctx.lineWidth = 22; ctx.lineCap = 'round'; ctx.lineJoin = 'round'; ctx.strokeStyle = '#000';
var drawing = false;
function pos(e) { var r = cnv.getBoundingClientRect(); return [e.clientX - r.left, e.clientY - r.top]; }
cnv.addEventListener('pointerdown', function(e) { drawing = true; var p = pos(e); ctx.beginPath(); ctx.moveTo(p[0], p[1]); });
cnv.addEventListener('pointermove', function(e) { if (!drawing) return; var p = pos(e); ctx.lineTo(p[0], p[1]); ctx.stroke(); });
window.addEventListener('pointerup', function() { drawing = false; });
document.getElementById('btn_clear').onclick = function() { ctx.clearRect(0, 0, 280, 280); };
var drawn_digit = new Promise(function(resolve) {
  document.getElementById('btn_done').onclick = function() {
    var d = ctx.getImageData(0, 0, 280, 280).data, s = '';
    for (var i = 0; i < 280 * 280; i++) { s += String.fromCharCode(d[i * 4 + 3]); }  // alpha channel = ink
    resolve(btoa(s));
  };
});
</script>
"""

try:
    from IPython.display import display, HTML
    from google.colab import output as _colab_output
    display(HTML(CANVAS_HTML))
    b64 = _colab_output.eval_js("drawn_digit", timeout_sec=600)
    canvas = np.frombuffer(base64.b64decode(b64), np.uint8).reshape(280, 280).astype(float) / 255.0
    show_prediction(net, to_mnist(canvas))
except ImportError:
    print("The drawing canvas only works in Google Colab.")
    print("Showing a random MNIST digit instead:")
    i = np.random.randint(X_test.shape[1])
    show_prediction(net, X_test[:, i], int(y_test[i]))

---
## Part 8. Break it

Now the most useful part: seeing what the result actually depends on. Change one number, run the cell below and see what happened to the accuracy.

| What to change | What to try | What should happen |
|---|---|---|
| size of the middle layers | `[784, 4, 4, 10]` | too few neurons to tell all the digits apart |
| number of layers | `[784, 16, 10]` | one middle layer instead of two — how much worse? |
| step size `eta` | `0.01` | tiny steps, learning almost stands still |
| step size `eta` | `30.0` | huge steps, the network jumps over the right spot |
| number of passes | `epochs=1` | how far does a single pass get you? |

In [ ]:
# change the numbers here and run the whole cell

my_sizes = [784, 16, 16, 10]
my_eta = 3.0
my_epochs = 5

experiment = init_network(my_sizes)
train(experiment, X_train, Y_train, epochs=my_epochs, batch_size=10, eta=my_eta)
print(f"\nResult: {accuracy(experiment, X_test, y_test)*100:.2f}%")

---
## What you have now

A working neural network built without a single machine learning library — just numpy, which is ordinary arithmetic on arrays of numbers. There is no PyTorch and no TensorFlow here.

And more importantly: what you just assembled is not a toy diagram, it is how real neural networks work. The big models differ in size, in the shape of their layers and in the tricks used to train them, but the two actions inside the loop stay the same: **work out which way to move the weights, and move them**.

**Want to write all the maths yourself?** There is a longer version of this same notebook where `backprop` and the rest are written from scratch from the formulas — [workshop_advanced.ipynb](https://colab.research.google.com/github/ewanpy00/Deeplearning_workshop/blob/main/workshop_advanced.ipynb).

**Where all this comes from:** the [3Blue1Brown video on neural networks](https://www.youtube.com/watch?v=aircAruvnKk) — the best visual explanation of what happens inside.